# TensorRT 기초: PyTorch → ONNX → TensorRT

> 실행 환경: Colab GPU 런타임 (VS Code: **Select Kernel → Colab → GPU**)
> study와 무관한 boilerplate(캐시, 추론 래퍼, 벤치마크)는 [`src/vopt`](../src/vopt)에 있습니다.

## 0. Setup
- Colab에 기본 설치된 torch는 그대로 사용하고, 없는 패키지만 `uv`로 설치합니다.
- 이 저장소를 clone해서 `vopt` 헬퍼를 import합니다. 로컬에서 `src/vopt`를 수정했다면 push한 뒤 이 셀을 다시 실행하세요(`git pull` + autoreload).

In [ ]:
# 0. Setup (Colab 런타임에서 실행)
import os, subprocess, sys

REPO_URL = "https://github.com/limJhyeok/vopt-learning.git"
REPO_DIR = "/content/vopt-learning"
PKGS = ["tensorrt-cu12", "onnx"]  # Colab 기본 torch/torchvision은 재설치하지 않음

assert os.path.isdir("/content"), "Colab 런타임이 아닙니다. 커널을 Colab으로 선택하세요."

def sh(cmd: str) -> None:
    print("$", cmd)
    subprocess.run(cmd, shell=True, check=True)

sh("pip install -q uv")
if os.path.isdir(REPO_DIR):
    sh(f"git -C {REPO_DIR} pull -q --ff-only")
else:
    sh(f"git clone -q --depth 1 {REPO_URL} {REPO_DIR}")
sh("uv pip install --system -q " + " ".join(PKGS))

if f"{REPO_DIR}/src" not in sys.path:
    sys.path.insert(0, f"{REPO_DIR}/src")

%reload_ext autoreload
%autoreload 2

from vopt.env import print_env
print_env()

> **(선택) Google Drive 캐시**: 명령 팔레트 → `Colab: Mount Google Drive to Server...`로 Drive를 마운트하면,
> 빌드한 engine이 `MyDrive/vopt-cache`에 저장되어 런타임이 바뀌어도 유지됩니다. (마운트하지 않으면 `/content/vopt-cache`)

## 1. PyTorch 모델 → ONNX

In [ ]:
import torch
import torchvision
from vopt import cache

model = torchvision.models.resnet18(weights="DEFAULT").eval().cuda()
x = torch.randn(1, 3, 224, 224, device="cuda")

onnx_path = cache.cache_dir() / "resnet18.onnx"
if not onnx_path.exists():
    # dynamo=False: 기존 TorchScript 기반 exporter. 에러가 나면 PKGS에 "onnxscript"를 추가하고 dynamo=True로 바꾸세요.
    torch.onnx.export(
        model, (x,), str(onnx_path),
        input_names=["input"], output_names=["logits"],
        opset_version=17, dynamo=False,
    )
print(onnx_path, f"{onnx_path.stat().st_size / 1e6:.1f} MB")

## 2. TensorRT engine 빌드
`get_engine`은 **로컬/Drive 캐시 → GitHub Release → 직접 빌드** 순서로 engine을 찾습니다.
파일명에 *ONNX 및 빌드 옵션 해시 + GPU 아키텍처 + TRT 버전*이 들어 있어서, 조건이 다르면 자동으로 다시 빌드합니다.
빌드 과정 자체를 공부할 때는 `force_rebuild=True`를 쓰거나 `trt_utils.build_engine`을 직접 보세요.

In [ ]:
from vopt import trt_utils

engine_path = trt_utils.get_engine("resnet18", onnx_path, fp16=True)
trt_model = trt_utils.TRTModule(engine_path)
print("inputs :", trt_model.input_names)
print("outputs:", trt_model.output_names)

## 3. 정확도 & 속도 비교

In [ ]:
from vopt.bench import benchmark, compare

with torch.inference_mode():
    ref = model(x)
    out = trt_model(x)
print("max abs diff:", (ref - out.float()).abs().max().item())
print("top-1 match :", (ref.argmax(1) == out.argmax(1)).all().item())

compare({
    "pytorch fp32": benchmark(lambda: model(x)),
    "tensorrt fp16": benchmark(lambda: trt_model(x)),
})

## 4. (선택) 빌드한 engine 공유하기
다음 study 때나 블로그 독자가 빌드를 건너뛸 수 있도록 GitHub Release에 올립니다.
1. 명령 팔레트 → `Colab: Download...`로 캐시 디렉토리의 `*.engine` 파일을 로컬 `artifacts/`에 받습니다.
2. 로컬에서 `scripts/publish_artifact.sh artifacts/*.engine`을 실행합니다.

engine은 **같은 GPU 아키텍처와 같은 TRT 버전**에서만 재사용할 수 있습니다(예: T4로 빌드한 engine은 L4에서 사용 불가).